In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip -q install segmentation-models-pytorch
import torch
import segmentation_models_pytorch as smp

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

m = smp.UnetPlusPlus("resnet34", encoder_weights="imagenet", classes=1).to("cuda")
x = torch.randn(8, 3, 256, 256, device="cuda")

with torch.no_grad():
    feats = m.encoder(x)
    out = m(x)

print("output shape:", tuple(out.shape))
for i, f in enumerate(feats):
    print("encoder feature", i, tuple(f.shape))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.2 MB/s eta 0:00:00
GPU available: True
GPU name: Tesla T4


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

output shape: (8, 1, 256, 256)
encoder feature 0 (8, 3, 256, 256)
encoder feature 1 (8, 64, 128, 128)
encoder feature 2 (8, 64, 64, 64)
encoder feature 3 (8, 128, 32, 32)
encoder feature 4 (8, 256, 16, 16)
encoder feature 5 (8, 512, 8, 8)


In [4]:
import torch, torch.nn as nn, torch.nn.functional as F
import segmentation_models_pytorch as smp

class AttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(g_ch, inter_ch, 1), nn.BatchNorm2d(inter_ch))
        self.Wx = nn.Sequential(nn.Conv2d(x_ch, inter_ch, 1), nn.BatchNorm2d(inter_ch))
        self.psi = nn.Sequential(nn.Conv2d(inter_ch, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):          # g = decoder signal, x = skip feature
        a = F.relu(self.Wg(g) + self.Wx(x))
        return x * self.psi(a)        # skip scaled by 0..1 weights

def conv_block(i, o, p):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Dropout2d(p))              # p=0 for now; dropout placement is decided in the protocol

class AttentionUNet(nn.Module):
    def __init__(self, encoder_name="resnet34", weights="imagenet", dec=(256,128,64,32,16), p=0.0):
        super().__init__()
        self.encoder = smp.encoders.get_encoder(encoder_name, in_channels=3, depth=5, weights=weights)
        enc = self.encoder.out_channels            # (3, 64, 64, 128, 256, 512)
        skips_ch = [enc[4], enc[3], enc[2], enc[1], 0]
        self.gates, self.blocks = nn.ModuleList(), nn.ModuleList()
        in_ch = enc[5]
        for s, d in zip(skips_ch, dec):
            self.gates.append(AttentionGate(in_ch, s, max(s // 2, 8)) if s > 0 else nn.Identity())
            self.blocks.append(conv_block(in_ch + s, d, p))
            in_ch = d
        self.head = nn.Conv2d(dec[-1], 1, 1)
    def forward(self, x):
        f = self.encoder(x)
        d = f[5]
        for gate, block, s in zip(self.gates, self.blocks, [f[4], f[3], f[2], f[1], None]):
            d = F.interpolate(d, scale_factor=2, mode="nearest")
            if s is not None:
                d = torch.cat([d, gate(d, s)], 1)
            d = block(d)
        return self.head(d)

a = AttentionUNet().cuda()
x = torch.randn(8, 3, 256, 256, device="cuda")
with torch.no_grad():
    y = a(x)
print("output shape:", tuple(y.shape))
print("Attention U-Net params (M):", round(sum(p.numel() for p in a.parameters()) / 1e6, 1))
u = smp.UnetPlusPlus("resnet34", encoder_weights=None, classes=1)
print("UNet++ params (M):", round(sum(p.numel() for p in u.parameters()) / 1e6, 1))

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

output shape: (8, 1, 256, 256)
Attention U-Net params (M): 24.6
UNet++ params (M): 26.1


In [5]:
!pip -q install grad-cam
!pip -q install segmentation-models-pytorch
import numpy as np, torch
import segmentation_models_pytorch as smp
from pytorch_grad_cam import LayerCAM
from pytorch_grad_cam.utils.model_targets import SemanticSegmentationTarget

def test_layercam(model, layer, name):
    model = model.cuda().eval()
    x = torch.randn(1, 3, 256, 256, device="cuda")
    with torch.no_grad():
        out = model(x)
    pred = (out[0, 0] > 0).float().cpu().numpy()      # predicted lesion pixels
    if pred.sum() == 0:                               # untrained model may predict nothing
        pred = np.ones_like(pred)
    targets = [SemanticSegmentationTarget(0, pred)]
    with LayerCAM(model=model, target_layers=[layer]) as cam:
        g = cam(input_tensor=x, targets=targets)
    print(name, "-> CAM shape:", g.shape, "| min/max:", float(g.min()), float(g.max()))

unetpp = smp.UnetPlusPlus("resnet34", encoder_weights=None, classes=1)
test_layercam(unetpp, unetpp.decoder, "UNet++")

attn = AttentionUNet(weights=None)
test_layercam(attn, attn.blocks[-1], "Attention U-Net")

UNet++ -> CAM shape: (1, 256, 256) | min/max: 0.0 0.9999998807907104
Attention U-Net -> CAM shape: (1, 256, 256) | min/max: 0.0 0.9999998807907104
